<a href="https://colab.research.google.com/github/KaiHaVertzz29/other-projects/blob/main/player_comparision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
pd.options.mode.chained_assignment = None
from scipy import stats
import requests
import json
from bs4 import BeautifulSoup
import statsmodels.api as sm
from functools import reduce


### Scraping data

In [ ]:
name = ['passing','gca','shooting','defense','misc','possession']

In [ ]:
for i in name:
  response = requests.get(f'https://fbref.com/en/comps/Big5/{i}/players/Big-5-European-Leagues-Stats')
  soup = BeautifulSoup(response.content,'html.parser')
  data_table = soup.find('table',id=f'stats_{i}')
  table_list = pd.read_html(str(data_table))
  df = table_list[0]
  df.columns = df.columns.droplevel(0)
  df.to_csv(f'{i}.csv')
  print(f'{i} done')

passing done
gca done
shooting done
defense done
misc done
possession done


### Loading data into dataframes and cleaning it

In [ ]:
defence = pd.read_csv('defense.csv')
gca = pd.read_csv('gca.csv')
misc = pd.read_csv('misc.csv')
passing = pd.read_csv('passing.csv')
possession = pd.read_csv('possession.csv')
shooting = pd.read_csv('shooting.csv')

In [ ]:
def clean_data(df):
  df.drop(['Unnamed: 0','Rk','Matches'],axis=1,inplace=True)
  df.drop(df[df.Player=='Player'].index,inplace=True)
  df.reset_index(drop=True,inplace=True)
  df[df.columns[5:]] = df[df.columns[5:]].astype('float64')
  df[df.columns[5:]] = df[df.columns[5:]].round(2)
  return df

In [ ]:
dataframes = [defence,gca,misc,passing,possession,shooting]

In [ ]:
for i in dataframes:
  i = clean_data(i)

### percentile function

In [ ]:
def get_percentile(df):

  df.drop(columns=['Comp','Pos'],inplace=True)
  needed_columns = df.columns[1:].tolist()

  for i in needed_columns:
    df[i+'_percentile'] = df[i].rank(pct=True)

  for i in needed_columns:
    df[i+'_rank'] = df[i].rank(ascending=False)

  df.reset_index(drop=True,inplace=True)

  df.drop_duplicates(subset='Player',inplace=True)
  df.reset_index(drop=True,inplace=True)

  return df

### Defense Function

In [ ]:
def get_defence_data(dataframes, player_name = None, defenders_only = False,League_only = False, limit = None):

  df = pd.DataFrame()
  d_passing = passing[['Player','Cmp%','Cmp%.1','Cmp%.2','Cmp%.3','1/3','PrgP','xAG','KP','xA']]
  d_creation = gca[['Player','90s','SCA90','GCA90']]
  d_duels = misc[['Player','Fls','Crs','Int','TklW','Recov','Won%']]
  d_possession = possession[['Player','Pos','Succ%','PrgC','1/3']]
  d_defence = defence[['Player','Comp','Def 3rd','Mid 3rd','Tkl%','Blocks','Clr','Err']]

  df = reduce(
      lambda left,right: pd.merge(left,right,on='Player'),
      [d_passing, d_creation, d_duels, d_possession, d_defence])

  if df[df.Player==player_name].empty:
    print('Player Not Found')
    return None

  if limit != None:
    df = df[df['90s']>=limit]

  league_name = df[df.Player==player_name]['Comp'].values[0]

  if League_only == True and defenders_only == True:

    def_league = df[(df.Comp==league_name) & (df.Pos.isin(['DF','DF,FW','DF,MF','MF,DF','FW,DF']))]
    def_league = get_percentile(def_league)
    return def_league, def_league[def_league.Player==player_name]

  elif League_only == True and defenders_only == False:

    league = df[df.Comp==league_name]
    league = get_percentile(league)
    return league, league[league.Player==player_name]

  elif defenders_only == True and League_only == False:

    defense_only = df[df.Pos.isin(['DF','DF,FW','DF,MF','MF,DF','FW,DF'])]
    defense_only = get_percentile(defense_only)
    return defense_only, defense_only[defense_only.Player==player_name]

  else:

    non = get_percentile(df)
    return non,non[non.Player==player_name]

### Attack Function

In [ ]:
def get_attack_data(dataframes, player_name = None, attackers_only = False,League_only = False, limit = None):

  df = pd.DataFrame()

  d_creation = gca[['Player', '90s', 'SCA90', 'GCA90','TO']]
  d_possession = possession[['Player', 'Pos', 'Succ%', 'PrgC', '1/3','Carries']]
  d_defence = defence[['Player','Err']]
  d_duels = misc[['Player','Comp','Crs','Won%']]
  d_passing = passing[['Player','Cmp%','Cmp%.1','Cmp%.2','Cmp%.3','1/3','PrgP','xAG','KP','xA','CrsPA']]
  d_shooting = shooting[['Player','SoT%','xG','npxG']]

  df = reduce(
      lambda left,right: pd.merge(left,right,on='Player'),
      [d_passing, d_creation, d_duels, d_possession, d_defence, d_shooting])

  df['Pa_Crs_percentage'] = df['CrsPA']/df['Crs']
  df.drop(columns=['CrsPA'],inplace=True)

  if df[df.Player==player_name].empty:
    print('Player Not Found')
    return None

  if limit != None:
    df = df[df['90s']>=limit]


  league_name = df[df.Player==player_name]['Comp'].values[0]

  if League_only == True and attackers_only == True:

    att_league = df[(df.Comp==league_name) & (df.Pos.isin(['FW','DF,FW','FW,MF','MF,FW','FW,DF']))].reset_index(drop=True)
    att_league = get_percentile(att_league)
    return att_league, att_league[att_league.Player==player_name]

  elif League_only == True and attackers_only == False:

    league = df[df.Comp==league_name]
    league = get_percentile(league)
    return league, league[league.Player==player_name]

  elif attackers_only == True and League_only == False:

    attack_only = df[df.Pos.isin(['FW','DF,FW','FW,MF','MF,FW','FW,DF'])].reset_index(drop=True)
    attack_only = get_percentile(attack_only)
    return attack_only, attack_only[attack_only.Player==player_name]

  else:

    non = get_percentile(df)
    return non,non[non.Player==player_name]

### as for

In [ ]:
x,y=get_attack_data(dataframes,player_name='Rodri',attackers_only=True,League_only=False,limit=12)

(430, 73)


In [ ]:
x[x.Player=='Federico Chiesa'][['Cmp%','Succ%']]

,Cmp%,Succ%
79,69.9,36.8


In [ ]:
x[x.Player=='Nico Williams'][['Player','Cmp%','Succ%']]

,Player,Cmp%,Succ%
414,Nico Williams,69.9,44.6


In [ ]:
x[x.Player=='Rafael Leão'][['Player','Cmp%','Succ%']]

,Player,Cmp%,Succ%
219,Rafael Leão,73.6,50.0


In [ ]:
passi = pd.read_csv('passing.csv')

In [ ]:
passi = clean_data(passi)

In [ ]:
passi.columns

Index(['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', '90s', 'Cmp',
       'Att', 'Cmp%', 'TotDist', 'PrgDist', 'Cmp.1', 'Att.1', 'Cmp%.1',
       'Cmp.2', 'Att.2', 'Cmp%.2', 'Cmp.3', 'Att.3', 'Cmp%.3', 'Ast', 'xAG',
       'xA', 'A-xAG', 'KP', '1/3', 'PPA', 'CrsPA', 'PrgP'],
      dtype='object')

In [ ]:
passi['Att_PrP'] = passi['PrgP']/passi['Att']

In [ ]:
need = passi[passi['90s']>=12]

In [ ]:
need['rank_pass'] = need['Att_PrP'].rank(ascending=False)

In [ ]:
need_mid = need[~(need.Pos.isin(['GK','DF','DF,MF','MF,DF','FW,DF','DF,FW']))]

In [ ]:
bell = need[['Player','Squad','Att','PrgP','Att_PrP','rank_pass']].sort_values('rank_pass')

In [ ]:
bell[bell.Player=='Jude Bellingham']

,Player,Squad,Att,PrgP,Att_PrP,rank_pass
265,Jude Bellingham,Real Madrid,1543.0,196.0,0.127025,78.0


In [ ]:
f = bell.groupby('Squad').Att_PrP.mean().reset_index()

In [ ]:
passi.columns

Index(['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', '90s', 'Cmp',
       'Att', 'Cmp%', 'TotDist', 'PrgDist', 'Cmp.1', 'Att.1', 'Cmp%.1',
       'Cmp.2', 'Att.2', 'Cmp%.2', 'Cmp.3', 'Att.3', 'Cmp%.3', 'Ast', 'xAG',
       'xA', 'A-xAG', 'KP', '1/3', 'PPA', 'CrsPA', 'PrgP', 'Att_PrP'],
      dtype='object')